#### **Install Dependencies**

In [1]:
! pip install -q splade-index[core] sentence-transformers datasets

#### **Load Dataset**

In [8]:
from datasets import load_dataset

# load Amharic Question Answering dataset
amharic_qa = load_dataset("israel/AmharicQA", split="validation").shuffle(seed=42)
amharic_qa

Dataset({
    features: ['context', 'question', 'answers'],
    num_rows: 595
})

In [9]:
queries = amharic_qa['question']
documents = amharic_qa['context']
answers = amharic_qa['answers']

len(queries), len(documents), len(answers)

(595, 595, 595)

#### **Build the search index with Amharic SPLADE**

In [ ]:
from sentence_transformers import SparseEncoder
from splade_index import SPLADE

# Download a SPLADE model from the 🤗 Hub
model = SparseEncoder("rasyosef/splade-amharic-base")

In [13]:
# Create your corpus here
corpus = [document for document in set(documents)]
print("Number of Documents:", len(corpus))

Number of Documents: 56


In [ ]:
# Create the SPLADE retriever and index the corpus
retriever = SPLADE()
retriever.index(model=model, documents=corpus)

#### **Query the Index**

In [14]:
idx = 6
query = queries[idx]
print(f"Query: {query}")

Query: በአጼ ቀዳማዊ ኃይለ ሥላሴ ታውጆ እስከ 1948 አ.ም. ሲያገለግል የነበረው ሕገ መንግሥት መች ነበር የታወጀው?


In [ ]:
# Query the corpus
query_list = [query]

# Get top-k results as a tuple of (doc ids, documents, scores). All three are arrays of shape (n_queries, k).
results = retriever.retrieve(query_list, k=3)
doc_ids, result_docs, scores = results.doc_ids, results.documents, results.scores

In [22]:
print("Top Results:")
for i in range(doc_ids.shape[1]):
    doc_id, doc, score = doc_ids[0, i], result_docs[0, i], scores[0, i]
    print(f"Rank {i+1} (score: {score:.2f}) (doc_id: {doc_id}): {doc}")
    print("-"*64)

Top Results:
Rank 1 (score: 26.97) (doc_id: 7): በ1240 አካባቢ በግብጽ አገር ቅብጡ አቡል ፋዳዒል እብን አል-አሣል ፍትሐ ነገሥት በአረብኛ ጻፉ። እብን አል-አሣል ሕጎቹን የወሰዱት በከፊል ከሃዋርያት ጽህፈቶችና ከሕገ ሙሴ፤ በከፊልም ከድሮ ቢዛንታይን ነገስታት ሕጎች ነበር። መጽሐፉ በግዕዝ ተተርጉሞ ወደ ኢትዮጵያ የገባበት ወቅት በዐፄ ዘርዐ ያዕቆብ ዘመን በ1450 አከባቢ እንደነበር የሚል ታሪካዊ መዝገብ አለ።  ቢሆንም መጀመርያ እንደ ሕገ ምንግሥት መጠቀሙን የተመዘገበው በዓፄ ሠርፀ ድንግል ዘመን (ከ1555 ጀምሮ) ነው። ፍትሐ ነገስት እስከ 1923 ዓ.ም. የብሄሩን ዋና ሕግ ሆኖ ቆየ። በ1904 ዓ.ም. ዐጼ 2 ምኒልክ የዘመናዊ ሕገ መንግሥት ፅንሰ ሀሳብ ተረድተው በጸሀፊው መምሬ ብስራት አንድ ሕገ መንግሥት የሚመስል ሰነድ ታተመ፤ ይህ ግን በውነት ሕገ መንግሥት አይባልም። «በምኒሊክ ስለተቋቋሙት ሚኒስቴሮች በምሳሌ የቀረበ ጽሑፍ» በምሳሌና በትርጓሜ የእያንዳንዱን ሚኒስቴር ሥራ እንደ ሰውነት አካላት አስመሰለ። ለምሳሌ፦ ኢትዮጵያ መጀመርያ ዘመናዊ ሕገ መንግሥት የተቀበለችው በ1923 አመተ ምኅረት በአጼ ቀዳማዊ ኃይለ ሥላሴ የታወጀ ሲሆን የሕግ አማካሪ ቤቶች በሁለት ያስከፈለ ነው።  ይህ የመንግሥታቸው ዋና መሠረት ሆኖ እስከ 1948 አ.ም. አገለገለ፤ በዚያ አመት ተሻሽሎ በወጣ ሕገ መንግሥት ሕዝቦች በመንግሥት ሥራ የሚጫወቱት ሚና እንደገና ተስፋፋ።  ይህ ብቻ የአገሪቱ ሕገ መንግሥት ሆኖ እስከ 1967 አ.ም. ቆየ፤ የዛኔ በደርግ (በሕገ መንግስት እራሱ ባልሆነ ሂደት) ተሰረዘ።
----------------------------------------------------------------
Rank 2 (score: 12.98) (doc_id: 10